In [ ]:
import os
import shutil
import zipfile
from pathlib import Path

from google.colab import drive, files

os.chdir("/content")
uploaded = files.upload()
if "smartvision_code.zip" not in uploaded:
    raise FileNotFoundError("Choose smartvision_code.zip in the upload dialog.")

extract_to = Path("/content/smartvision_upload")
if extract_to.exists():
    shutil.rmtree(extract_to)
extract_to.mkdir()
with zipfile.ZipFile("/content/smartvision_code.zip") as archive:
    archive.extractall(extract_to)

matches = [
    path.parents[1]
    for path in extract_to.rglob("prepare_dataset.py")
    if path.parent.name == "scripts" and (path.parents[1] / "requirements.txt").is_file()
]
if not matches:
    raise FileNotFoundError("smartvision_code.zip has no scripts/prepare_dataset.py")
root = matches[0]
os.chdir(root)
print("project", root)

!pip install -q -r requirements.txt

drive.mount("/content/drive")
drive_runs = Path("/content/drive/MyDrive/smartvision_coe/runs")
drive_runs.mkdir(parents=True, exist_ok=True)
runs = root / "outputs" / "runs"
runs.parent.mkdir(parents=True, exist_ok=True)
if runs.is_symlink():
    if runs.resolve() != drive_runs.resolve():
        runs.unlink()
        runs.symlink_to(drive_runs, target_is_directory=True)
elif runs.exists():
    for item in runs.iterdir():
        destination = drive_runs / item.name
        if not destination.exists():
            shutil.move(str(item), destination)
    shutil.rmtree(runs)
    runs.symlink_to(drive_runs, target_is_directory=True)
else:
    runs.symlink_to(drive_runs, target_is_directory=True)
print("training runs", drive_runs)

from ultralytics import YOLO

if not getattr(YOLO.train, "_smartvision_resume", False):
    _train = YOLO.train

    def train(self, *args, **kwargs):
        project = Path(kwargs.get("project", drive_runs))
        name = kwargs.get("name", "colab")
        last = project / str(name) / "weights" / "last.pt"
        if last.is_file() and not kwargs.get("resume"):
            print(f"resuming from {last}")
            kwargs["resume"] = str(last)
        return _train(self, *args, **kwargs)

    train._smartvision_resume = True
    YOLO.train = train


# SmartVision training on a Colab GPU

A 50-epoch YOLOv8s run is about 53 hours on the Mac CPU. This notebook repeats the dataset merge and trains on a GPU.

1. **Runtime → Change runtime type → GPU** (T4 or better).
2. Upload this repository as a zip to `/content` (leave out `.venv`, `data/raw`, and `data/merged`), then unzip it so `scripts/prepare_dataset.py` is on disk.
3. In the left sidebar, open **Secrets** and add `ROBOFLOW_API_KEY`. Do not paste the key into a cell.


In [ ]:
!pip install -q ultralytics roboflow python-dotenv pyyaml


In [ ]:
import os
from pathlib import Path

from google.colab import userdata

os.environ["ROBOFLOW_API_KEY"] = userdata.get("ROBOFLOW_API_KEY")

candidates = [
    Path.cwd(),
    Path("/content/smartvision_coe"),
    Path("/content"),
]
root = next(path for path in candidates if (path / "scripts" / "prepare_dataset.py").is_file())
os.chdir(root)
print("project", root.resolve())


In [ ]:
from pathlib import Path

from scripts.prepare_dataset import (
    build_merged_dataset,
    download_datasets,
    format_report,
    load_config,
)

root = Path.cwd()
config = load_config(root / "config" / "dataset_classes.yaml")
download_datasets(config, root / "data" / "raw")
stats = build_merged_dataset(config, root / "data" / "raw", root / "data" / "merged")
print(format_report(stats))


In [ ]:
from scripts.train import main

main(["--device", "cuda", "--name", "colab"])


In [ ]:
from google.colab import files

files.download("models/best.pt")
